<a href="https://colab.research.google.com/github/AlmeidaLadson/auditius-voice-assistant/blob/main/AudITus_VoiceAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -r requirements.txt --quiet
print("Dependências instaladas com sucesso.")

In [ ]:
from google.colab import userdata

AUDITIUS_GROQ_TOKEN = userdata.get('AUDITIUS_GROQ_TOKEN')
print("Chave AudITus carregada com segurança.")

In [ ]:
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode

AUDITIUS_RECORD_SECONDS = 6

AUDITIUS_JS = """
(async () => {
  const sleep = (ms) => new Promise(r => setTimeout(r, ms));
  const toBase64 = (buf) => {
    let s = '';
    new Uint8Array(buf).forEach(b => s += String.fromCharCode(b));
    return btoa(s);
  };

  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream, { mimeType: 'audio/webm' });
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  console.log('🎙️ AudITus gravando...');
  await sleep(6000);
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  const blob = new Blob(chunks, { type: 'audio/webm' });
  const buf = await blob.arrayBuffer();
  const audio = toBase64(buf);
  google.colab.kernel.invokeFunction('auditius.receive_audio', [audio], {});
})();
"""

auditius_audio_bytes = None

def auditius_receive_audio(b64):
    global auditius_audio_bytes
    auditius_audio_bytes = b64decode(b64)

output.register_callback('auditius.receive_audio', auditius_receive_audio)
display(Javascript(AUDITIUS_JS))

In [ ]:
AUDITIUS_AUDIO_PATH = "auditius_input.webm"

import time

# Aguarda até que o áudio seja recebido do navegador
while auditius_audio_bytes is None:
    print("Aguardando gravação do áudio...", end="\r")
    time.sleep(0.5)
print("Áudio gravado!          ") # Limpa a linha de 'aguardando'

with open(AUDITIUS_AUDIO_PATH, "wb") as f:
    f.write(auditius_audio_bytes)

print(f"Áudio salvo em: {AUDITIUS_AUDIO_PATH}")

In [ ]:
from groq import Groq

auditius_client = Groq(api_key=AUDITIUS_GROQ_TOKEN)

# Transcrição via Whisper
with open(AUDITIUS_AUDIO_PATH, "rb") as f:
    auditius_transcricao = auditius_client.audio.transcriptions.create(
        model="whisper-large-v3",
        file=f,
        language="pt"
    )

auditius_pergunta = auditius_transcricao.text
print(f"🎤 Você perguntou: {auditius_pergunta}")

# Resposta via LLaMA com contexto de TI
auditius_resposta = auditius_client.chat.completions.create(
    model="llama-3.1-8b-instant", # Modelo atualizado para um disponível
    messages=[
        {
            "role": "system",
            "content": (
                "Você é o AudITus, um assistente de voz especializado em suporte técnico de TI. "
                "Responda de forma clara, objetiva e em português. "
                "Foque em soluções práticas para problemas de infraestrutura, redes, Linux e Windows Server."
            )
        },
        {"role": "user", "content": auditius_pergunta}
    ]
)

auditius_texto_resposta = auditius_resposta.choices[0].message.content
print(f"\n🤖 AudITus: {auditius_texto_resposta}")

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

AUDITIUS_AUDIO_RESPOSTA = "auditius_output.mp3"

auditius_voz = gTTS(text=auditius_texto_resposta, lang="pt", slow=False)
auditius_voz.save(AUDITIUS_AUDIO_RESPOSTA)

print("🔊 Reproduzindo resposta do AudITus...")
display(Audio(AUDITIUS_AUDIO_RESPOSTA, autoplay=True))